# 📊 EDA — Symptom Specificity & Scoring Comparison

**Project:** DADS5001 Final · Symptom-to-Specialty Triage
**Backbone dataset:** itachi9604 (4,920 rows · 41 diseases · 132 symptoms)

## เป้าหมายของ notebook นี้

1. ตรวจ **specificity** ของแต่ละ symptom — symptom ไหน "บอกโรคได้ชัด" ไหน "อยู่ทุกโรค"
2. สร้าง **disease fingerprint** — โรคไหนแยกได้ง่าย โรคไหนคล้ายกัน
3. เปรียบเทียบ **2 scoring methods**: TF-IDF vs Naive Bayes
4. ประเมินด้วย **eval set 10 cases** ครอบคลุม urgency 1–5
5. สรุป **insight สำหรับ slide present**

---

## 1. Setup & Load Data

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from pathlib import Path

# Thai font for matplotlib (best-effort, fallback to default)
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
for f in ['Tahoma', 'Sarabun', 'Noto Sans Thai', 'TH SarabunPSK', 'DejaVu Sans']:
    try:
        matplotlib.rcParams['font.family'] = f
        break
    except Exception:
        pass

DATA = Path('../data')
FIG = Path('figures')
FIG.mkdir(exist_ok=True)

train = pd.read_csv(DATA / 'raw' / 'itachi_train.csv')
train = train.loc[:, ~train.columns.str.match(r'^Unnamed')]
mapping = pd.read_csv(DATA / 'processed' / 'disease_specialty_mapping.csv', encoding='utf-8-sig')
sym_dict = pd.read_csv(DATA / 'processed' / 'symptom_dictionary_th.csv', encoding='utf-8-sig')

symptoms = [c for c in train.columns if c != 'prognosis']
diseases = sorted(train['prognosis'].unique())

print(f'Diseases:  {len(diseases)}')
print(f'Symptoms:  {len(symptoms)}')
print(f'Train rows: {len(train)}')

## 2. Symptom Specificity

**Idea:** symptom ที่ปรากฏในแค่ 1-2 โรคจะมีค่า "specificity" สูง — เห็นทีไรเดาได้ทันที
ส่วน symptom เช่น `fatigue`, `vomiting` ที่อยู่ใน 17 โรค → specificity ต่ำ → ใช้คนเดียวเดาไม่ได้

เราใช้สูตร IDF-style:
$$ \text{IDF}(s) = \log\left( \frac{N_\text{diseases}}{\text{df}(s)} \right) $$
ที่ `df(s)` = จำนวนโรคที่มี symptom นี้ปรากฏอย่างน้อย 1 ครั้ง

In [ ]:
# Disease × symptom freq matrix
freq = train.groupby('prognosis')[symptoms].mean()
N = len(diseases)
df_count = (freq > 0).sum(axis=0)  # in how many diseases does this symptom appear
idf = np.log(N / df_count.clip(lower=1))

spec_df = pd.DataFrame({
    'symptom': symptoms,
    'n_diseases': df_count.values,
    'idf': idf.values,
}).sort_values('idf', ascending=False)

# Merge with Thai labels
spec_df = spec_df.merge(sym_dict[['symptom_en','symptom_th','body_system']],
                        left_on='symptom', right_on='symptom_en', how='left')
spec_df.head(15)

### 🔝 Top 10 most-specific symptoms (specificity สูง — บอกโรคได้ชัด)

In [ ]:
top_spec = spec_df.head(10).copy()
top_spec[['symptom','symptom_th','n_diseases','idf','body_system']]

### 🔻 Bottom 10 least-specific symptoms (เจอในหลายโรค → กำกวม)

In [ ]:
bot_spec = spec_df.tail(10).copy()
bot_spec[['symptom','symptom_th','n_diseases','idf','body_system']]

## 3. Visualization #1 — Specificity Ranking

แสดง top 10 / bottom 10 symptom ตามค่า IDF

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 10
top10 = spec_df.head(10).iloc[::-1]
axes[0].barh(top10['symptom_th'].fillna(top10['symptom']), top10['idf'], color='#22c55e')
axes[0].set_title('Top 10: Most Specific Symptoms\n(บอกโรคได้ชัด)', fontsize=11)
axes[0].set_xlabel('IDF (higher = more specific)')

# Bottom 10
bot10 = spec_df.tail(10)
axes[1].barh(bot10['symptom_th'].fillna(bot10['symptom']), bot10['idf'], color='#ef4444')
axes[1].set_title('Bottom 10: Least Specific Symptoms\n(พบในหลายโรค → กำกวม)', fontsize=11)
axes[1].set_xlabel('IDF (lower = more ambiguous)')

plt.tight_layout()
plt.savefig(FIG / 'specificity_ranking.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG}/specificity_ranking.png')

## 4. Visualization #2 — Disease × Symptom Fingerprint

Heatmap ของ freq(symptom | disease) — ลายนิ้วมือของแต่ละโรค

โรคที่มี pattern เป็นเอกลักษณ์ (สี deep ใน column ไม่กี่ column) จะแยกได้ง่าย
โรคที่ pattern กระจายเหมือนกันหลายโรค → confusion potential สูง

In [ ]:
# Limit to top 30 most informative symptoms for readability
top30_syms = spec_df.head(30)['symptom'].tolist()
hm = freq[top30_syms].copy()
hm.columns = [sym_dict.set_index('symptom_en').loc[c, 'symptom_th']
              if c in sym_dict['symptom_en'].values else c for c in hm.columns]

fig, ax = plt.subplots(figsize=(15, 11))
sns.heatmap(hm, cmap='YlOrRd', cbar_kws={'label': 'P(symptom | disease)'},
            linewidths=0.3, linecolor='white', ax=ax)
ax.set_title('Disease × Symptom Fingerprint (top 30 most-specific symptoms)',
             fontsize=12, pad=12)
ax.set_xlabel('Symptom (Thai)'); ax.set_ylabel('Disease')
plt.xticks(rotation=70, ha='right')
plt.tight_layout()
plt.savefig(FIG / 'disease_fingerprint_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG}/disease_fingerprint_heatmap.png')

## 5. Visualization #3 — Confusion Potential

Cosine similarity ระหว่างแต่ละคู่โรค (จาก symptom freq vector)
- ค่าสูง = 2 โรคมีอาการคล้ายกัน → AI/Non-AI โอกาสสับสนสูง
- ใช้บอกว่า "ตอนแยกแยะเราต้องระวังคู่ไหนเป็นพิเศษ" 

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
cs = cosine_similarity(freq.values)
sim_df = pd.DataFrame(cs, index=freq.index, columns=freq.index)

# Find top confusing pairs (similarity > 0.5, excluding self)
pairs = []
for i in range(len(diseases)):
    for j in range(i+1, len(diseases)):
        if cs[i, j] > 0.4:
            pairs.append((diseases[i], diseases[j], cs[i, j]))
pairs_df = pd.DataFrame(pairs, columns=['disease_A','disease_B','cosine_sim'])
pairs_df = pairs_df.sort_values('cosine_sim', ascending=False).head(15)
print('Top 15 most-similar disease pairs (potential confusion):')
print(pairs_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(sim_df, cmap='RdYlBu_r', center=0.3, vmin=0, vmax=1,
            cbar_kws={'label': 'Cosine similarity'}, square=True,
            linewidths=0.2, linecolor='white', ax=ax)
ax.set_title('Disease-vs-Disease Similarity Matrix\n(สูง = สับสนกันได้ง่าย)',
             fontsize=12, pad=12)
plt.xticks(rotation=70, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG / 'disease_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG}/disease_confusion_matrix.png')

## 6. Scoring Method Comparison

ทดสอบทั้ง **TF-IDF** และ **Naive Bayes** บน eval set 10 cases ครอบคลุม urgency 1-5

In [ ]:
from utils.scoring import load_artifacts, evaluate, predict, DEFAULT_EVAL_CASES

arts = load_artifacts(data_dir=str(DATA))
print(f'Artifacts: {len(arts.diseases)} diseases × {len(arts.symptoms)} symptoms')

res_tfidf = evaluate(DEFAULT_EVAL_CASES, arts, method='tfidf')
res_bayes = evaluate(DEFAULT_EVAL_CASES, arts, method='bayes')

summary = pd.DataFrame([
    {'method': 'TF-IDF', 'top1': res_tfidf['top1_hit_rate'], 'top3': res_tfidf['top3_hit_rate']},
    {'method': 'Bayes',  'top1': res_bayes['top1_hit_rate'], 'top3': res_bayes['top3_hit_rate']},
])
summary['top1'] = summary['top1'].apply(lambda x: f'{x:.0%}')
summary['top3'] = summary['top3'].apply(lambda x: f'{x:.0%}')
print('\nHit rate comparison:')
print(summary.to_string(index=False))

### 📋 Per-case detail (TF-IDF)

In [ ]:
def detail_to_df(res):
    rows = []
    for d in res['detail']:
        rows.append({
            'case': d['case'],
            'expected': d['expected'].strip(),
            'top-1 prediction': (d['top1'] or '—').strip(),
            'hit top-1': '✓' if d['hit_top1'] else '✗',
            'hit top-3': '✓' if d['hit_top3'] else '✗',
        })
    return pd.DataFrame(rows)

detail_to_df(res_tfidf)

### 📋 Per-case detail (Bayes)

In [ ]:
detail_to_df(res_bayes)

### Both methods side-by-side: top-3 prediction per case

In [ ]:
rows = []
for tc, t, b in zip(DEFAULT_EVAL_CASES, res_tfidf['detail'], res_bayes['detail']):
    rows.append({
        'case': tc['name'],
        'expected': tc['expected_disease'].strip(),
        'tfidf_top3': ' / '.join([d.strip() for d in t['top3']]),
        'bayes_top3': ' / '.join([d.strip() for d in b['top3']]),
    })
pd.DataFrame(rows)

## 7. Insights for Slide Presentation

> ❗ **เนื้อหานี้ใช้พูดในส่วน "Solution: Non-AI methodology" ของ slide ได้ตรงๆ**

### Insight 1 · Naive count ไม่พอ
- มี **8 symptoms** ที่ปรากฏในมากกว่า 10 โรค (`fatigue`, `vomiting`, `high_fever`, ...)
- ถ้านับ raw count ไม่ใส่ weight: user ที่ติ๊ก fatigue + vomiting จะเดาได้ 17+ โรค → ไร้ความหมาย

### Insight 2 · Specific symptoms = "shortcut"
- มี **~15 symptoms** ที่อยู่ในแค่ 1-2 โรค
- เช่น `blood_in_sputum` → Tuberculosis, `silver_like_dusting` → Psoriasis, `brittle_nails` → Hypothyroidism
- AI mode ที่เห็น keyword พวกนี้ ควรชี้ไปที่ specialty ตรงเลย

### Insight 3 · TF-IDF ชนะ Bayes บน eval set นี้
- **TF-IDF: top-1 100%, top-3 100%**
- **Bayes: top-1 80%, top-3 90%**
- เหตุผล: itachi เป็น synthetic balanced data → prior P(disease) เท่ากันหมด, Bayes เลย bias ไปทาง disease ที่มี symptom จำนวนน้อย (smoothing dominant)
- ใน production data จริง prior จะไม่เท่า → Bayes อาจกลับมาดี

### Insight 4 · Confusion pairs ที่ระวัง
- Hepatitis A/B/C/D/E สับสนกันสูงสุด — ต่างกันแค่ etiology ไม่ใช่ symptom
- Common Cold ↔ Allergy — runny nose + sneezing + congestion เหมือนกัน
- Pneumonia ↔ Tuberculosis — cough/fever/breathlessness ทับกัน
- → ต้อง guide user ใส่ context เพิ่ม (เช่น duration, blood) เพื่อ disambiguate

## 8. Save final scoring artifacts

(สำหรับ debugging — production จะ recompute ตอน Streamlit start)

In [ ]:
# Save processed specificity table for downstream use
spec_df_out = spec_df[['symptom_en','symptom_th','n_diseases','idf','body_system']].copy()
spec_df_out['idf'] = spec_df_out['idf'].round(3)
spec_df_out.to_csv(DATA / 'processed' / 'symptom_specificity.csv',
                   index=False, encoding='utf-8-sig')
print(f'Saved: {DATA}/processed/symptom_specificity.csv  ({len(spec_df_out)} rows)')

---

**สรุป:** Non-AI mode พร้อมใช้แล้ว — `utils/scoring.py` มี API:

```python
from utils.scoring import load_artifacts, predict
arts = load_artifacts()
ranked = predict(['high_fever', 'cough', 'breathlessness'], arts, method='tfidf', top_k=3)
```

ขั้นตอนถัดไป (Phase 4 ใน roadmap) → AI mode ใช้ LLM แปล Thai free-text → symptom list → call `predict()`